#### Cleaning Section: [cleaning_report.md](../docs/Cleaning_logs/cleaning_report.md)

- `Early Data Analysis`
- `Duplicate Correction`
- `Inconsistencies Fixing`
- `Missing Data Checking`
- `Planning for EDA`

In [53]:
# Data was given in snippets, we have to concat everything into one df
import pandas as pd
import plotly.express as px

years = list(range(2020, 2027))
dfs = []

for year in years:
    for quarter in ["Q1", "Q2", "Q3", "Q4"]:
        file_path = f"../data/raw/BDD PRODUCCION/{year}/{quarter} {year}.csv"
        try:
            df = pd.read_csv(file_path)

            if year == 2025 and quarter == "Q2":
                s = df["order_date"]
                p1 = pd.to_datetime(s, dayfirst=True, errors="coerce")
                p2 = pd.to_datetime(s, dayfirst=False, errors="coerce")
                num = pd.to_numeric(s, errors="coerce")
                excel = pd.to_datetime(num, unit="D", origin="1899-12-30")
                # Fill order_date with p2 first, then p1, and finally excel if both are NaT 
                df["order_date"] = p2.fillna(p1).fillna(excel)
            else:
                df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

            dfs.append(df)
        except FileNotFoundError as e:
            print(f"File not found: {file_path}, error: {e}")

remissions = pd.concat(dfs, ignore_index=True)
remissions.head()


File not found: ../data/raw/BDD PRODUCCION/2026/Q2 2026.csv, error: [Errno 2] No such file or directory: '../data/raw/BDD PRODUCCION/2026/Q2 2026.csv'
File not found: ../data/raw/BDD PRODUCCION/2026/Q3 2026.csv, error: [Errno 2] No such file or directory: '../data/raw/BDD PRODUCCION/2026/Q3 2026.csv'
File not found: ../data/raw/BDD PRODUCCION/2026/Q4 2026.csv, error: [Errno 2] No such file or directory: '../data/raw/BDD PRODUCCION/2026/Q4 2026.csv'


,Year,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
0,2020,51013232,2020-02-04,1073,2020-02-04 07:00:00,7044,510,6.0,2020-02-04 06:45:13,2020-02-04 08:17:24,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
1,2020,51013233,2020-02-04,1073,2020-02-04 07:00:00,6603,510,5.5,2020-02-04 06:45:22,2020-02-04 08:26:06,101,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
2,2020,51013235,2020-02-04,1028,2020-02-04 08:00:00,9631,510,2.5,2020-02-04 07:37:56,2020-02-04 08:55:49,78,ONE TIME ESP ANGELICA RIVERA GOMEZ,ROMERO JULIA,CALLE MANUEL BECERRA 14515 COL ALAMEDAS,CH-F5
3,2020,51013236,2020-02-04,1005,2020-02-04 08:15:00,6598,510,3.0,2020-02-04 08:02:36,2020-02-04 09:47:43,105,IVAN NOE SIMENTAL ORTEGA,COLECTOR SACRAMENTO,ARROLLO MIMBRE Y VIALIDAD SACRAMENTO,CH-J9
4,2020,51013238,2020-02-04,1026,2020-02-04 08:30:00,10145,510,3.5,2020-02-04 08:09:01,2020-02-04 09:09:15,60,ONE TIME CONSTRUCENTRO CHIH,MOLINA BALDERRAMA CESAR,PERIF DE LA JUVENTUD 9926 COL RESIDENCIA,CH-L5


DO NOT TOUCH THIS CODE (Fixes issue with format of `Q2 2025.csv`)

In [54]:
order_date = pd.to_datetime(remissions["order_date"], errors="coerce").dt.normalize()

def rebuild_datetime(col):
    raw = remissions[col].astype(str).str.strip()

    time_str = raw.str.extract(r"(\d{1,2}:\d{2}(?::\d{2})?\s*[APMapm]{0,2})")[0]
    time_str = time_str.str.replace(r"\.\d+", "", regex=True)

    t24 = pd.to_datetime(time_str, format="%H:%M:%S", errors="coerce")
    t24 = t24.fillna(pd.to_datetime(time_str, format="%H:%M", errors="coerce"))

    t12 = pd.to_datetime(time_str, format="%I:%M:%S %p", errors="coerce")
    t12 = t12.fillna(pd.to_datetime(time_str, format="%I:%M %p", errors="coerce"))

    t = t24.fillna(t12)
    time_only = t - t.dt.normalize()

    return order_date + time_only

remissions["typed_time"] = rebuild_datetime("typed_time")
remissions["start_time"] = rebuild_datetime("start_time")
remissions["at_plant_time"] = rebuild_datetime("at_plant_time")

In [55]:
# show rows from May 2025 
remissions_may = remissions[(remissions["order_date"].dt.month == 5) & (remissions["order_date"].dt.year == 2025)]
remissions_may.head()

,Year,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
295158,2025,51008733,2025-05-02,1212,2025-05-02 07:00:00,7043,510,3.5,2025-05-02 06:15:15,2025-05-02 08:05:51,110,ONE TIME OCTAVIO RIOS,NORMA CHAPARRO VAZQUEZ,CALLE BOSQUE DE PIEDRA #2828 BOSQUE DE,CHG-2
295159,2025,51008735,2025-05-02,1223,2025-05-02 07:00:00,9431,510,3.5,2025-05-02 06:15:52,2025-05-02 07:55:40,100,CASA HOGAR PARA NIÑOS YIREH,CASA HOGAR PARA NIÑOS YIREH,BAHIA DE SAN QUINTIN 9513 COL RINCONADAS,CHK-3
295160,2025,51008737,2025-05-02,1227,2025-05-02 07:15:00,6605,510,3.0,2025-05-02 06:27:16,2025-05-02 08:33:59,126,LEONEL RAMIREZ JARAMILLO,LEONEL RAMIREZ JARAMILLO,CIRCUITO PODERES #10702 SAN GABRIEL ETA,CHD-4
295161,2025,51008739,2025-05-02,1423,2025-05-02 07:45:00,12193,510,6.5,2025-05-02 07:47:03,2025-05-02 09:17:53,90,JONATHAN MARQUEZ BACA,OBRAS VARIAS,CALLE ENRIQUE MULLER & LEONA VICARIO RE,CHK-5
295162,2025,51008740,2025-05-02,1214,2025-05-02 08:15:00,9431,510,2.0,2025-05-02 07:55:56,2025-05-02 08:37:36,42,INDUSTRIAS CALCITE,AV TECNOLOGICO,AV TECNOLOGICO 11708 REVOLUCION CHIHU,CHJ-6


#### Checking issue with some data corruption from plant 710 

In [56]:
# Diagnose how Q2 2025 dates are parsed around May 2-3 vs Apr 6-7 for p1 (raw q2) and p2 (remmissions)
remissions_window = remissions.loc[
    (remissions["order_date"] >= "2025-04-01")
    & (remissions["order_date"] < "2025-05-10"),
    "order_date"
].dt.date.value_counts().sort_index()
remissions_window.head(40)

raw_q2_2025 = pd.read_csv("../data/raw/BDD PRODUCCION/2025/Q2 2025.csv")
raw_q2_2025["order_date_raw"] = raw_q2_2025["order_date"].astype(str).str.strip()

raw_q2_2025["p1"] = pd.to_datetime(raw_q2_2025["order_date_raw"], dayfirst=True, errors="coerce")
raw_q2_2025["p2"] = pd.to_datetime(raw_q2_2025["order_date_raw"], dayfirst=False, errors="coerce")
raw_q2_2025["num"] = pd.to_numeric(raw_q2_2025["order_date_raw"], errors="coerce")
raw_q2_2025["excel"] = pd.to_datetime(raw_q2_2025["num"], unit="D", origin="1899-12-30")

def sample_range(series, start, end):
    mask = (series >= start) & (series < end)
    cols = ["order_date_raw", "p1", "p2", "excel"]
    return raw_q2_2025.loc[mask, cols].head(20)

print("p1 May 2-3:")
display(sample_range(raw_q2_2025["p1"], "2025-05-02", "2025-05-04"))

print("p2 May 2-3:")
display(sample_range(raw_q2_2025["p2"], "2025-05-02", "2025-05-04"))

print("excel May 2-3:")
display(sample_range(raw_q2_2025["excel"], "2025-05-02", "2025-05-04"))

print("p1 Apr 6-7:")
display(sample_range(raw_q2_2025["p1"], "2025-04-06", "2025-04-08"))

print("p2 Apr 6-7:")
display(sample_range(raw_q2_2025["p2"], "2025-04-06", "2025-04-08"))

print("excel Apr 6-7:")
display(sample_range(raw_q2_2025["excel"], "2025-04-06", "2025-04-08"))

p1 May 2-3:


,order_date_raw,p1,p2,excel


p2 May 2-3:


,order_date_raw,p1,p2,excel
0,5/2/2025,2025-02-05,2025-05-02,NaT
1,5/2/2025,2025-02-05,2025-05-02,NaT
2,5/2/2025,2025-02-05,2025-05-02,NaT
3,5/2/2025,2025-02-05,2025-05-02,NaT
4,5/2/2025,2025-02-05,2025-05-02,NaT
5,5/2/2025,2025-02-05,2025-05-02,NaT
6,5/2/2025,2025-02-05,2025-05-02,NaT
7,5/2/2025,2025-02-05,2025-05-02,NaT
8,5/2/2025,2025-02-05,2025-05-02,NaT
9,5/2/2025,2025-02-05,2025-05-02,NaT


excel May 2-3:


...

,order_date_raw,p1,p2,excel
1091,6/4/2025,2025-04-06,2025-06-04,NaT
1092,6/4/2025,2025-04-06,2025-06-04,NaT
1093,6/4/2025,2025-04-06,2025-06-04,NaT
1094,6/4/2025,2025-04-06,2025-06-04,NaT
1095,6/4/2025,2025-04-06,2025-06-04,NaT
1096,6/4/2025,2025-04-06,2025-06-04,NaT
1097,6/4/2025,2025-04-06,2025-06-04,NaT
1098,6/4/2025,2025-04-06,2025-06-04,NaT
1099,6/4/2025,2025-04-06,2025-06-04,NaT
1100,6/4/2025,2025-04-06,2025-06-04,NaT


p2 Apr 6-7:


,order_date_raw,p1,p2,excel


excel Apr 6-7:


,order_date_raw,p1,p2,excel


1. Variable Formatting

In [57]:
# Change 'order_date' and 'typed_time' to datetime format
remissions['order_date'] = pd.to_datetime(remissions['order_date'], errors='raise', format='mixed')
remissions['typed_time'] = pd.to_datetime(remissions['typed_time'], errors='raise', format='mixed')
remissions['start_time'] = pd.to_datetime(remissions['start_time'], errors='raise', format='mixed')
remissions['at_plant_time'] = pd.to_datetime(remissions['at_plant_time'], errors='raise', format='mixed')
remissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 356332 entries, 0 to 356331
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Year                 356332 non-null  int64         
 1   tkt_code             356332 non-null  int64         
 2   order_date           356332 non-null  datetime64[us]
 3   order_code           356332 non-null  int64         
 4   start_time           356332 non-null  datetime64[us]
 5   truck_code           356332 non-null  int64         
 6   ship_plant_code      356332 non-null  int64         
 7   u_Volumen            356332 non-null  float64       
 8   typed_time           356332 non-null  datetime64[us]
 9   at_plant_time        356332 non-null  datetime64[us]
 10  u_Cicle              356332 non-null  int64         
 11  name                 356331 non-null  str           
 12  Nombre del proyecto  356332 non-null  str           
 13  ship_addr_line       3562

In [58]:
# Change `ship_plant_code`, `tkt_code`, `order_code` and `truck_code` to string type
remissions['ship_plant_code'] = remissions['ship_plant_code'].astype(str)
remissions['tkt_code'] = remissions['tkt_code'].astype(str)
remissions['order_code'] = remissions['order_code'].astype(str)
remissions['truck_code'] = remissions['truck_code'].astype(str)
remissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 356332 entries, 0 to 356331
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Year                 356332 non-null  int64         
 1   tkt_code             356332 non-null  str           
 2   order_date           356332 non-null  datetime64[us]
 3   order_code           356332 non-null  str           
 4   start_time           356332 non-null  datetime64[us]
 5   truck_code           356332 non-null  str           
 6   ship_plant_code      356332 non-null  str           
 7   u_Volumen            356332 non-null  float64       
 8   typed_time           356332 non-null  datetime64[us]
 9   at_plant_time        356332 non-null  datetime64[us]
 10  u_Cicle              356332 non-null  int64         
 11  name                 356331 non-null  str           
 12  Nombre del proyecto  356332 non-null  str           
 13  ship_addr_line       3562

In [59]:
# We don't need column `Year`
remissions = remissions.drop(columns=["Year"]) # One-time use
remissions.head(1)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
0,51013232,2020-02-04,1073,2020-02-04 07:00:00,7044,510,6.0,2020-02-04 06:45:13,2020-02-04 08:17:24,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1


Null/Missing data check

In [60]:
# Check for null values in each column
nulls = remissions.isnull().sum()
nulls = nulls[nulls > 0]  # keep only columns with at least 1 null

null_summary = pd.DataFrame({
    "null_count": nulls,
    "null_%": (nulls / len(remissions) * 100).round(2)
}).sort_values("null_count", ascending=False)

null_summary

,null_count,null_%
map_page,444,0.12
ship_addr_line,85,0.02
name,1,0.00


In [61]:
# It´s okay to drop rows with null values (<2%)
remissions = remissions[remissions['ship_addr_line'].notnull() & remissions['map_page'].notnull() & remissions['name'].notnull()]
print("Number of rows after dropping null values:", remissions.shape[0])

Number of rows after dropping null values: 355802


Duplicate Check

In [62]:
# Duplicate check for all time columns
typed_time_duplicates = remissions[remissions.duplicated(subset=['start_time', 'at_plant_time','truck_code'], keep=False)]
typed_time_duplicates = typed_time_duplicates.sort_values(by='typed_time', ascending=True)
typed_time_duplicates.head(10)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page


In [63]:
print("Number of duplicate rows based on start_time, at_plant_time, and truck_code:", typed_time_duplicates.shape[0])

Number of duplicate rows based on start_time, at_plant_time, and truck_code: 0


In [64]:
# It is impossible for many trucks to have the same start_time and at_plant_time
# Therefore, these are duplicates that should be removed, as they are likely to be errors in the data entry process.
remissions = remissions.drop(typed_time_duplicates.index)
print("Number of rows after dropping duplicates:", remissions.shape[0])

Number of rows after dropping duplicates: 355802


##### Outlier and Missing data Checking

In [65]:
# Check missing data over time
# Analyze hourly distribution of u_Volumen by ship_plant_code
hourly_distribution = (
    remissions
    .assign(hour_bucket=remissions["start_time"].dt.floor("h"))
    .groupby(["hour_bucket", "ship_plant_code"], as_index=False)["u_Volumen"]
    .sum()
)

fig = px.line(
    hourly_distribution,
    x="hour_bucket",
    y="u_Volumen",
    color="ship_plant_code",
    title="Hourly Distribution of u_Volumen by Ship Plant Code (2020-2026)"
)
fig.show()

In [66]:
# Analyze hourly distribution of remission count by ship_plant_code
hourly_distribution = (
    remissions
    .assign(hour_bucket=remissions["start_time"].dt.floor("h"))
    .groupby(["hour_bucket", "ship_plant_code"], as_index=False)
    .size()
    .rename(columns={"size": "remission_count"})
)

hourly_distribution["u_Volumen"] = hourly_distribution["remission_count"]

fig = px.line(
    hourly_distribution,
    x="hour_bucket",
    y="remission_count",
    color="ship_plant_code",
    title="Hourly Distribution of Remissions by Ship Plant Code (2020-2026)"
)
fig.show()

#### General Missing Data Observations:
- The common no-data-registered periods for all plants are:
    + `January 1st 2022` - `Feb 2nd 2022`
        + **Most likely reason**: Winter Vacations
    + `December 31 2023` - `Feb 1st 2024`
        + **Most likely reason**: Winter Vacations
    + `Mar 31 2025` - `May 2nd 2025`
    
- There are NO apparent stop from working during `2022-2023` winter vacations period nor `2024-2025`

- **Action taken:** After finishing cleaning phase I will apply imputation methods based on historic data and similar plants only for non-winter-vacations gaps (which is one). [Data_Imputation](./2.data_imputation.ipynb). As for winter gaps, probably make a winter-vacations categorical column.

#### Missing Data per plant and Outliers:
- Plant 514
    + Begins to register since `July 20 2023` 
    --- **Action Taken** ---
    + No action, TFT is not affected for this kind of "late" data

- Plant 710
    + Massive gap between `May 2023` - `Feb 2024`
    + Outlier during `June 2021`
    --- **Action Taken** ---
    + After cleaning, I will apply imputation methods based on historic data and similar plants. [Data_Imputation](./2.data_imputation.ipynb)

- Plant 512
    + Heavy downfall in remissions and volume between `April 2020` and `June 15 2020` most likely due to pandemic
    --- **Action Taken** ---
    + Because it is the only plant with this downturn, I will add a `plant-specific regime flag` for this specific plant for the model to know this event (pandemmic)
    + Outlier during `September 2024`

- Plant 510:
    + `October 2021` outlier

- Plant 511:
    + `April 2026` outlier
- Plant 717:
    + Too few data and non-operational since 2022
    --- **Action Taken** ---
    + Check percentage of data and consider deleting the whole plant

- Plant 515
    + `May 2024` and `June 2025` Outliers




#### Other Observations
- `April 2025` has no data in the given raw dataset, but there are rows for `April 6` and `April 7` of said year, also data from `May 2` and `May 3` is not appearing.
- Outliers to check:
    + Too many orders and volume in `Sept 5 2024 10:00am`
    + Some outliers for late 2025

Deleting Plant 717

In [67]:
# See percentage of plant 717 remissions count 
plant_717_count = remissions[remissions["ship_plant_code"] == "717"].shape[0]
total_count = remissions.shape[0]
print(f"Percentage of remissions from plant 717: {plant_717_count / total_count * 100:.2f}%")

Percentage of remissions from plant 717: 0.56%


In [68]:
# Get rid of plant 717 remissions since they are only 0.5% of the data and have a very different pattern than the rest of the plants, which could be due to data entry errors or a different process that is not representative of the other plants.
remissions = remissions[remissions["ship_plant_code"] != "717"]

Outlier Fixing

In [69]:
# Plant 710 remission count boxplot (row count) 
plant_710_counts = (
    remissions.loc[remissions["ship_plant_code"] == "710"]
    .groupby(remissions["start_time"].dt.floor("h"))
    .size()
    .reset_index(name="remission_count")
)

fig = px.box(
    plant_710_counts,
    y="remission_count",
    title="Boxplot of Hourly Remission Count for Plant 710",
)
fig.show()


In [70]:
# Plant 710 amount of times that remission count has been above the upper fence (Q3 + 1.5*IQR)
Q1 = plant_710_counts["remission_count"].quantile(0.25)
Q3 = plant_710_counts["remission_count"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

outliers = plant_710_counts[plant_710_counts["remission_count"] > upper_fence]
print(f"Number of hours where remission count for plant 710 is above the upper fence: {outliers.shape[0]}")

outlier_table = (
    outliers["remission_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)
outlier_table.columns = ["remission_count", "hour_count"]

display(outlier_table)

Number of hours where remission count for plant 710 is above the upper fence: 245


,remission_count,hour_count
0,10,75
1,11,46
2,12,22
3,13,22
4,14,13
5,15,11
6,16,4
7,17,6
8,18,2
9,19,1


In [71]:
# Only actual outlier looking ad the graph would be the 16 remissions of June 14th 2021 at 10am
# See all remissions from June 14th 2021 at 10am (10:00-10:59) to check for unusual repetitions
outlier_remissions = remissions[
    (remissions["ship_plant_code"] == "710") &
    (remissions["start_time"] >= "2021-06-14 10:00:00") &
    (remissions["start_time"] < "2021-06-14 11:00:00")
]
outlier_remissions.head(16)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
75428,71089278,2021-06-14,1037,2021-06-14 10:15:00,6302,710,2.0,2021-06-14 10:09:48,2021-06-14 11:39:11,90,ONE TIME CARLOS GALVAN,HERRERA ROJAS BENJAMIN,PEDRO ROBLES 6323 COL SAUCITO,CH-N5
75429,71089279,2021-06-14,1041,2021-06-14 10:30:00,6600,710,1.5,2021-06-14 10:11:01,2021-06-14 11:08:00,57,CONSTRUCCIONES PROFESIONALES GAMA,FRAC TARRAGONA II ( CTU ),FRAC TARRAGONA II ( CTU ) S/N FRAC T,CH-N3
75437,71089290,2021-06-14,1055,2021-06-14 10:45:00,4392,710,6.0,2021-06-14 10:32:06,2021-06-14 13:04:28,152,OSCAR LEONEL TOCORNAL CARRILLO,TOCORNAL CARRILLO OSCAR LEONEL,PEDREGAL DEL ALBA CALLE ALBA DE TOMRMES,CH-N2
75439,71089294,2021-06-14,1084,2021-06-14 10:00:00,9625,710,1.0,2021-06-14 10:36:47,2021-06-14 11:30:23,54,ONE TIME PROMOCIONES ANGELICA R,TORRES ALBA,FUENTE TREVI 6601 LAS FUENTES,CH-N1
75440,71089295,2021-06-14,1118,2021-06-14 10:15:00,4417,710,1.5,2021-06-14 10:42:52,2021-06-14 11:30:46,48,CONSTRUCCIONES PROFESIONALES GAMA,FRAC TRENTO II ( CTU ),FRAC TRENTO II ( CTU ) S/N FRAC TRENT,CH-N2


We'll keep plant 710's "outliers" due to legitimate data and no clues of repetition or errors

In [72]:
# Plant 510 remission count boxplot (row count) 
plant_510_counts = (
    remissions.loc[remissions["ship_plant_code"] == "510"]
    .groupby(remissions["start_time"].dt.floor("h"))
    .size()
    .reset_index(name="remission_count")
)
fig = px.box(
    plant_510_counts,
    y="remission_count",
    title="Boxplot of Hourly Remission Count for Plant 510",
)
fig.show()

In [73]:
# Plant 510 amount of times that remission count has been above the upper fence (Q3 + 1.5*IQR)
Q1 = plant_510_counts["remission_count"].quantile(0.25)
Q3 = plant_510_counts["remission_count"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

outliers = plant_510_counts[plant_510_counts["remission_count"] > upper_fence]
print(f"Number of hours where remission count for plant 510 is above the upper fence: {outliers.shape[0]}")

outlier_table = (
    outliers["remission_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)
outlier_table.columns = ["remission_count", "hour_count"]

display(outlier_table)

Number of hours where remission count for plant 510 is above the upper fence: 487


,remission_count,hour_count
0,11,168
1,12,89
2,13,63
3,14,40
4,15,23
5,16,16
6,17,14
7,18,7
8,19,6
9,20,5


In [74]:
# The 68 remission spike in plant 510 seems 

In [75]:
# Only actual outlier looking ad the graph would be the 16 remissions of October 5th 2021 at 9am
# See all remissions from October 5th 2021 at 9am (09:00-09:59) to check for unusual repetitions
outlier_remissions = remissions[
    (remissions["ship_plant_code"] == "510") &
    (remissions["start_time"] >= "2020-07-13 12:00:00") &
    (remissions["start_time"] < "2020-07-13 13:00:00")
]
outlier_remissions.head(21)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
13264,51021316,2020-07-13,1059,2020-07-13 12:00:00,10157,510,2.5,2020-07-13 11:15:40,2020-07-13 12:14:33,59,SANDOVAL MEDINA JORGE ALBERTO,OBRAS VARIAS,RINCONADAS DE LA SIERRA RINCONADA SENDER,CH-L3
13268,51021321,2020-07-13,1024,2020-07-13 12:00:00,9631,510,6.0,2020-07-13 11:42:53,2020-07-13 12:55:16,73,CESAR ALEJANDRO OSOLLO SAENZ,OSOLLO SAENZ CESAR ALEJANDRO,CIRCUITO MOLINO POYATOS #650 MOLINO DE V,CH-K6
13269,51021322,2020-07-13,1219,2020-07-13 12:30:00,9430,510,5.5,2020-07-13 12:11:23,2020-07-13 13:36:20,85,CONSEJO DE URBANIZACION MUNICIPAL D,C. DE LAS PALMERAS,DE LAS PALMERAS S/N FRACC QUINTAS DEL,CH-L9
13270,51021323,2020-07-13,1107,2020-07-13 12:00:00,10159,510,5.0,2020-07-13 12:41:19,2020-07-13 13:46:07,65,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-P1
13271,51021324,2020-07-13,1107,2020-07-13 12:00:00,10157,510,5.0,2020-07-13 12:43:02,2020-07-13 13:57:14,74,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-P1
13272,51021325,2020-07-13,1106,2020-07-13 12:00:00,9636,510,6.0,2020-07-13 13:12:42,2020-07-13 14:08:53,56,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-P1
13273,51021326,2020-07-13,1106,2020-07-13 12:00:00,6611,510,6.0,2020-07-13 13:13:09,2020-07-13 14:16:23,63,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-P1
13274,51021329,2020-07-13,1106,2020-07-13 12:00:00,7046,510,6.0,2020-07-13 13:32:58,2020-07-13 14:33:36,61,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-P1
13275,51021330,2020-07-13,1106,2020-07-13 12:00:00,9631,510,6.0,2020-07-13 13:45:26,2020-07-13 14:45:13,60,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-P1
13276,51021331,2020-07-13,1106,2020-07-13 12:00:00,9430,510,6.0,2020-07-13 13:57:22,2020-07-13 14:52:00,55,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-P1


In [76]:
# Plant 511 remission count boxplot (row count) 
plant_511_counts = (
    remissions.loc[remissions["ship_plant_code"] == "511"]
    .groupby(remissions["start_time"].dt.floor("h"))
    .size()
    .reset_index(name="remission_count")
)
fig = px.box(
    plant_511_counts,
    y="remission_count",
    title="Boxplot of Hourly Remission Count for Plant 511",
)
fig.show()

In [77]:
# Plant 511 amount of times that remission count has been above the upper fence (Q3 + 1.5*IQR)
Q1 = plant_511_counts["remission_count"].quantile(0.25)
Q3 = plant_511_counts["remission_count"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

outliers = plant_511_counts[plant_511_counts["remission_count"] > upper_fence]
print(f"Number of hours where remission count for plant 511 is above the upper fence: {outliers.shape[0]}")

outlier_table = (
    outliers["remission_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)
outlier_table.columns = ["remission_count", "hour_count"]

display(outlier_table)

Number of hours where remission count for plant 511 is above the upper fence: 246


,remission_count,hour_count
0,10,94
1,11,51
2,12,31
3,13,19
4,14,3
5,15,6
6,16,4
7,17,4
8,18,5
9,20,2


In [78]:
# See outlier remissions
outlier_remissions = remissions[
    (remissions["ship_plant_code"] == "511") &
    (remissions["start_time"] >= "2024-12-21 6:00:00") &
    (remissions["start_time"] < "2024-12-21 7:00:00")
]
outlier_remissions.head(22)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
270947,51147060,2024-12-21,1070,2024-12-21 06:00:00,9432,511,5.0,2024-12-21 00:02:13,2024-12-21 02:15:18,133,"CALIDAD, CONSTRUCCION Y CONSULTORIA",GRUPO BAFAR,CARR A CUAUHTEMOC Y C SANTA EULALIA S/N,CHW-6
270948,51147061,2024-12-21,1070,2024-12-21 06:00:00,6598,511,5.0,2024-12-21 00:12:27,2024-12-21 02:02:39,110,"CALIDAD, CONSTRUCCION Y CONSULTORIA",GRUPO BAFAR,CARR A CUAUHTEMOC Y C SANTA EULALIA S/N,CHW-6
270949,51147062,2024-12-21,1070,2024-12-21 06:00:00,9628,511,5.0,2024-12-21 00:23:38,2024-12-21 02:24:07,121,"CALIDAD, CONSTRUCCION Y CONSULTORIA",GRUPO BAFAR,CARR A CUAUHTEMOC Y C SANTA EULALIA S/N,CHW-6
270950,51147063,2024-12-21,1070,2024-12-21 06:00:00,13022,511,5.0,2024-12-21 00:36:59,2024-12-21 02:46:53,130,"CALIDAD, CONSTRUCCION Y CONSULTORIA",GRUPO BAFAR,CARR A CUAUHTEMOC Y C SANTA EULALIA S/N,CHW-6
270951,51147064,2024-12-21,1070,2024-12-21 06:00:00,6164,511,5.0,2024-12-21 00:47:16,2024-12-21 02:29:51,102,"CALIDAD, CONSTRUCCION Y CONSULTORIA",GRUPO BAFAR,CARR A CUAUHTEMOC Y C SANTA EULALIA S/N,CHW-6
270952,51147065,2024-12-21,1070,2024-12-21 06:00:00,13040,511,5.0,2024-12-21 00:54:07,2024-12-21 02:55:26,121,"CALIDAD, CONSTRUCCION Y CONSULTORIA",GRUPO BAFAR,CARR A CUAUHTEMOC Y C SANTA EULALIA S/N,CHW-6
270953,51147067,2024-12-21,1070,2024-12-21 06:00:00,13007,511,5.0,2024-12-21 01:16:25,2024-12-21 02:55:29,99,"CALIDAD, CONSTRUCCION Y CONSULTORIA",GRUPO BAFAR,CARR A CUAUHTEMOC Y C SANTA EULALIA S/N,CHW-6
270954,51147068,2024-12-21,1070,2024-12-21 06:00:00,6600,511,5.0,2024-12-21 01:30:33,2024-12-21 03:23:08,113,"CALIDAD, CONSTRUCCION Y CONSULTORIA",GRUPO BAFAR,CARR A CUAUHTEMOC Y C SANTA EULALIA S/N,CHW-6
270955,51147069,2024-12-21,1070,2024-12-21 06:00:00,13051,511,5.0,2024-12-21 01:40:14,2024-12-21 03:03:20,83,"CALIDAD, CONSTRUCCION Y CONSULTORIA",GRUPO BAFAR,CARR A CUAUHTEMOC Y C SANTA EULALIA S/N,CHW-6
270956,51147070,2024-12-21,1070,2024-12-21 06:00:00,9625,511,5.0,2024-12-21 01:49:01,2024-12-21 03:38:30,109,"CALIDAD, CONSTRUCCION Y CONSULTORIA",GRUPO BAFAR,CARR A CUAUHTEMOC Y C SANTA EULALIA S/N,CHW-6


In [79]:
# Plant 515 remission count boxplot (row count) 
plant_515_counts = (
    remissions.loc[remissions["ship_plant_code"] == "515"]
    .groupby(remissions["start_time"].dt.floor("h"))
    .size()
    .reset_index(name="remission_count")
)

fig = px.box(
    plant_515_counts,
    y="remission_count",
    title="Boxplot of Hourly Remission Count for Plant 515",
)
fig.show()


In [80]:
# Plant 515 amount of times that remission count has been above the upper fence (Q3 + 1.5*IQR)
Q1 = plant_515_counts["remission_count"].quantile(0.25)
Q3 = plant_515_counts["remission_count"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

outliers = plant_515_counts[plant_515_counts["remission_count"] > upper_fence]
print(f"Number of hours where remission count for plant 515 is above the upper fence: {outliers.shape[0]}")

outlier_table = (
    outliers["remission_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)
outlier_table.columns = ["remission_count", "hour_count"]

display(outlier_table)

Number of hours where remission count for plant 515 is above the upper fence: 147


,remission_count,hour_count
0,10,71
1,11,23
2,12,9
3,13,10
4,14,5
5,15,6
6,16,1
7,17,3
8,18,3
9,20,1


In [81]:
# Only actual outlier looking ad the graph would be the 16 remissions of June 14th 2025 at 1pm
# See all remissions from June 14th 2025 at 1pm (13:00-13:59) to check for unusual repetitions
outlier_remissions = remissions[
    (remissions["ship_plant_code"] == "515") &
    (remissions["typed_time"] >= "2025-06-14 13:00:00") &
    (remissions["typed_time"] < "2025-06-14 14:00:00")
]
outlier_remissions.head(21)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
307399,51583492,2025-06-14,1218,2025-06-14 08:15:00,6163,515,1.0,2025-06-14 13:03:57,2025-06-14 13:04:50,1,JOSE ABRAHAM RODRIGUEZ MORALES,RUBA MORTERO BARDAS FRACC SAN,FRACC SAN AGUSTIN BARDAS SN FRACC SAN,CHN-21
307400,51583493,2025-06-14,1235,2025-06-14 09:15:00,6163,515,1.0,2025-06-14 13:05:02,2025-06-14 13:16:21,11,COMERCIALIZADORA PASO DEL NORTE DE,FRAC PASEO DE LAS FLORES,PASEO GRANO DE ORO Y PASEO DEL SUIZO S/,CHQ-17
307401,51583494,2025-06-14,1221,2025-06-14 08:15:00,12829,515,1.0,2025-06-14 13:05:52,2025-06-14 13:17:10,12,RUBA DESARROLLOS,LUIS VELEZ JARDINES DE SAN AGUSTUN,LUIS VELEZ JARDINES DE SAN AGUSTIN 2 SN,CHR-21
307402,51583495,2025-06-14,1167,2025-06-14 09:15:00,12832,515,1.5,2025-06-14 13:09:43,2025-06-14 13:12:36,3,RUBA DESARROLLOS,CONSTRUCTER SAN AGUSTIN II BARDAS D,CONSTRUCTER SAN AGUSTIN II BARDAS DUPLEX,CHR-21
307403,51583500,2025-06-14,1256,2025-06-14 12:45:00,10154,515,3.0,2025-06-14 13:14:20,2025-06-14 13:19:50,5,EDGAR DURAN VALENZUELA,EDGAR DURAN VALENZUELA,PUERTA DE CHIHUAHUA,CHW-20
307404,51583501,2025-06-14,1168,2025-06-14 09:30:00,13219,515,1.0,2025-06-14 13:14:30,2025-06-14 13:17:46,3,RUBA DESARROLLOS,FABRINFRA JARDINES DE SAN AGUSTIN 2,FABRINFRA JARDINES DE SAN AGUSTIN 2 47 V,CHR-21
307405,51583502,2025-06-14,1220,2025-06-14 09:45:00,6600,515,3.0,2025-06-14 13:16:07,2025-06-14 13:23:17,7,RUBA DESARROLLOS,JOSE RODRIGUEZ SAN AGUSTIN 2 E2,SAN AGUSTIN 2 E2 BARDAS PERIM SN SAN,CHN-21
307406,51583503,2025-06-14,1173,2025-06-14 10:00:00,6163,515,3.5,2025-06-14 13:16:42,2025-06-14 13:24:24,8,RUBA DESARROLLOS,FABRINFRA JARDINES DE SAN AGUSTIN 2,FABRINFRA JARDINES DE SAN AGUSTIN 2 47 V,CHR-21
307407,51583504,2025-06-14,1222,2025-06-14 10:15:00,12829,515,1.0,2025-06-14 13:17:21,2025-06-14 13:23:35,6,RUBA DESARROLLOS,LIZBETH SALGADO SAN AGUSTIN 2 E2 BA,LIZBETH SALGADO SAN AGUSTIN 2 E2 BARDAS,CHR-21
307408,51583505,2025-06-14,1119,2025-06-14 10:30:00,13219,515,4.5,2025-06-14 13:17:54,2025-06-14 13:27:15,10,ONE TIME PROMOCIONES HUGO T,JORGE HERNANDEZ,VALLE DORADO,CHN-21


In [82]:
# Plant 512 remission count boxplot (row count) 
plant_512_counts = (
    remissions.loc[remissions["ship_plant_code"] == "512"]
    .groupby(remissions["start_time"].dt.floor("h"))
    .size()
    .reset_index(name="remission_count")
)

fig = px.box(
    plant_512_counts,
    y="remission_count",
    title="Boxplot of Hourly Remission Count for Plant 512",
)
fig.show()


In [83]:
# Plant 512 amount of times that remission count has been above the upper fence (Q3 + 1.5*IQR)
Q1 = plant_512_counts["remission_count"].quantile(0.25)
Q3 = plant_512_counts["remission_count"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

outliers = plant_512_counts[plant_512_counts["remission_count"] > upper_fence]
print(f"Number of hours where remission count for plant 512 is above the upper fence: {outliers.shape[0]}")

outlier_table = (
    outliers["remission_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)
outlier_table.columns = ["remission_count", "hour_count"]

display(outlier_table)

Number of hours where remission count for plant 512 is above the upper fence: 117


,remission_count,hour_count
0,14,42
1,15,23
2,16,9
3,17,8
4,18,4
5,19,4
6,20,3
7,21,1
8,22,7
9,23,2


**Why I will not cut this outliers?**: Because this phenomenons are real, I will not introduce false data, so I would rather use something like TransformedTargetRegressor0 `(https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html)` to reduce the weight of spikes like this during the training process.

---
##### Inconsistencies Check

In [84]:
remissions.describe()

,order_date,start_time,u_Volumen,typed_time,at_plant_time,u_Cicle
count,353797,353797,353797.000000,353797,353797,353797.000000
mean,2023-05-01 14:18:41.015724,2023-05-02 01:54:03.998694,3.186710,2023-05-02 01:42:30.083960,2023-05-02 02:59:18.489483,77.137443
min,2020-02-04 00:00:00,2020-02-04 05:00:00,0.340000,2020-02-04 05:30:26,2020-02-04 06:44:47,1.000000
25%,2021-10-21 00:00:00,2021-10-21 08:00:00,1.500000,2021-10-21 07:18:07,2021-10-21 08:56:44,56.000000
50%,2023-05-18 00:00:00,2023-05-18 16:00:00,3.000000,2023-05-18 15:41:03,2023-05-18 16:56:47,74.000000
75%,2024-11-04 00:00:00,2024-11-04 15:30:00,5.000000,2024-11-04 14:06:02,2024-11-04 15:19:09,96.000000
max,2026-04-24 00:00:00,2026-04-24 11:30:00,6.500000,2026-04-24 10:47:56,2026-04-24 10:49:00,209.000000
std,NaN,NaN,1.784885,NaN,NaN,35.836594


In [85]:
# Graph the distribution of u_Cicle to see if there are orders that were not completed or any unreal value
fig = px.histogram(remissions, x='u_Cicle', nbins=len(remissions['u_Cicle'].unique()), title='Distribution of u_Cicle')
fig.show()

In [86]:
# See amount of remissions per plant with u_Cicle equal to 1 (include percentage)
cicle_1_counts = remissions[remissions['u_Cicle'] == 1]['ship_plant_code'].value_counts().reset_index()
cicle_1_counts.columns = ['ship_plant_code', 'count']
total_cicle1 = cicle_1_counts['count'].sum()
cicle_1_counts['percentage'] = (cicle_1_counts['count'] / total_cicle1 * 100).round(2)
print(f"Total remissions with u_Cicle equal to 1: {total_cicle1}")
display(cicle_1_counts)

Total remissions with u_Cicle equal to 1: 9872


,ship_plant_code,count,percentage
0,512,4220,42.75
1,515,1715,17.37
2,514,1458,14.77
3,710,1372,13.90
4,511,783,7.93
5,510,324,3.28


In [87]:
# So many u_Cicle values at one, lets check the percentage of rows in which at_plant_time - typed_time for the rows with u_Cicle at 1 gives us the same value (which is 1)
cicle_1_remissions = remissions[remissions['u_Cicle'] == 1].copy()
cicle_1_remissions['time_diff'] = (cicle_1_remissions['at_plant_time'] - cicle_1_remissions['typed_time']).dt.total_seconds() / 60
cicle_1_remissions['time_diff_rounded'] = cicle_1_remissions['time_diff'].round(2)
matching_time_diff = cicle_1_remissions[cicle_1_remissions['time_diff_rounded'] == 1].shape[0]
total_cicle1 = cicle_1_remissions.shape[0]
print(f"Percentage of rows with u_Cicle equal to 1 where at_plant_time - typed_time is approximately 1 minute: {matching_time_diff / total_cicle1 * 100:.2f}%")

# Show me the rows that correspond to the rows with u_Cicle equal to 1 and at_plant_time - typed_time approximately 1 minute, to see if there are any patterns or if they correspond to a specific plant or time period
matching_rows = cicle_1_remissions[cicle_1_remissions['time_diff_rounded'] == 1]
print(f"Total rows with diff equal to 1: {matching_time_diff}")
matching_rows.head(20)

Percentage of rows with u_Cicle equal to 1 where at_plant_time - typed_time is approximately 1 minute: 0.84%
Total rows with diff equal to 1: 83


,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page,time_diff,time_diff_rounded
6538,51255316,2020-03-28,1084,2020-03-28 10:30:00,9799,512,1.5,2020-03-28 14:05:51,2020-03-28 14:06:51,1,RUBA DESARROLLOS,ALVAREZ ALTARIA CASAS MUESTRA RESID,ALVAREZ FRACC ALTARIA CASAS MUESTRA PROL,CH-V3,1.0,1.0
16570,51255821,2020-05-14,1079,2020-05-14 10:00:00,6603,512,1.0,2020-05-14 06:41:18,2020-05-14 06:42:18,1,GRUPO LOGISTICO Y SOLUCIONES INTEGR,ESTRADA CERRADA ALBERTA VISTAS PRAD,ESTRADA ALBERTA VISTAS PRADO 26 VIV,CH-B1,1.0,1.0
16593,51255856,2020-05-20,1166,2020-05-20 15:00:00,9799,512,4.5,2020-05-20 17:57:08,2020-05-20 17:58:08,1,MARIO EDUARDO LOPEZ CHAVEZ,LOPEZ CHAVEZ MARIO EDUARDO,ramon cordoba y rio conchos colonia revo,CH-H5,1.0,1.0
16684,51256055,2020-06-15,1335,2020-06-15 15:15:00,10147,512,1.5,2020-06-15 14:52:00,2020-06-15 14:53:00,1,GRUPO LOGISTICO Y SOLUCIONES INTEGR,ESTRADA CERRADA ALBERTA VISTAS PRAD,ESTRADA ALBERTA VISTAS PRADO 26 VIV,CH-B3,1.0,1.0
17044,51256597,2020-06-29,1153,2020-06-29 08:00:00,9627,512,2.0,2020-06-29 06:37:39,2020-06-29 06:38:39,1,GRUPO LOGISTICO Y SOLUCIONES INTEGR,CISCO MONTEVERDE 20 VIVIENDAS,CISCO MONTEVERDE 20 VIVIENDAS CISCO,CH-V3,1.0,1.0
17339,51256949,2020-07-06,1202,2020-07-06 09:45:00,6614,512,1.0,2020-07-06 11:05:15,2020-07-06 11:06:15,1,COPACHISA,INACTIVA CESSNA SUPRA,AV DEMING PARQUE INDUSTRIAL SUPRA ENSEG,CH-D3,1.0,1.0
17864,51257684,2020-07-17,1254,2020-07-17 14:15:00,6164,512,1.5,2020-07-17 14:04:10,2020-07-17 14:05:10,1,GRUPO LOGISTICO Y SOLUCIONES INTEGR,PALACIOS VISTAS PRADO 20 VIV,PALACIOS VISTAS PRADO 20 VIV S/N PALA,CH-B1,1.0,1.0
18448,51258483,2020-07-29,1312,2020-07-29 13:00:00,6599,512,2.0,2020-07-29 12:30:45,2020-07-29 12:31:45,1,GRUPO LOGISTICO Y SOLUCIONES INTEGR,ALBACE FRACC VISTAS PRADO 25 VIV,ALBACE FRACC VISTAS PRADO 25 VIV S/N,CH-B1,1.0,1.0
20456,71072523,2020-05-16,1096,2020-05-16 11:00:00,6300,710,1.0,2020-05-16 10:29:11,2020-05-16 10:30:11,1,GRUPO LOGISTICO Y SOLUCIONES INTEGR,ARQCR MONTEVERDE 22 VIVIENDAS,ARQCR MONTEVERDE 22 VIVIENDAS ARQCR,CH-V3,1.0,1.0
29064,51260152,2020-08-22,1064,2020-08-22 07:00:00,10148,512,5.0,2020-08-22 09:53:01,2020-08-22 09:54:01,1,LUIS ALEJANDRO RAMIREZ ROMERO,RAMIREZ ROMERO LUIS ALEJANDRO,PROLONGACION RIO URUGUAY ESQ VALLE DE ME,CH-B2,1.0,1.0


In [88]:
# Delete the rows with u_Cicle equal to 1 and at_plant_time - typed_time approximately 1 minute, since they are likely to be errors in the data entry process and they represent a very small percentage of the data (0.5%), which is not enough to affect the overall analysis but could introduce noise and bias if they are kept.
print(f"Number of rows before dropping u_Cicle 1 with time diff of 1 minute: {remissions.shape[0]}" )
mask = (
	(remissions['u_Cicle'] == 1) &
	(((remissions['at_plant_time'] - remissions['typed_time']).dt.total_seconds() / 60).round(2) == 1)
)
remissions = remissions[~mask]
print("Number of rows after dropping u_Cicle 1 with time diff of 1 minute:", remissions.shape[0])

Number of rows before dropping u_Cicle 1 with time diff of 1 minute: 353797
Number of rows after dropping u_Cicle 1 with time diff of 1 minute: 353714


In [89]:
# Boxplot of u_Cicle to check for outliers
fig = px.box(remissions, y='u_Cicle', title='Boxplot of u_Cicle')
fig.show()

##### Volume distribution check for outliers

In [90]:
# Calculate counts and percentages for u_Volumen
counts = remissions['u_Volumen'].value_counts().sort_index()
percentages = (counts / counts.sum()) * 100

# Histogram of frequency of each unique volume value to check for outliers
x_labels = counts.index.astype(str)  # treat volumes as categorical so bars are wider
fig = px.bar(x=x_labels, y=counts.values, title='Distribution of Volume per Truck',
             labels={'x':'u_Volumen','y':'count'}, text=counts.values)
fig.update_traces(
    textposition='outside',
    texttemplate='%{y}',
    customdata=percentages.values,
    hovertemplate='<b>Volume:</b> %{x}<br><b>Count:</b> %{y}<br><b>Percentage:</b> %{customdata:.2f}%<extra></extra>'
)
fig.update_layout(
    bargap=0.1,        # reduce gap between bars
    xaxis_tickangle=45,
    width=1000
)
fig.show()

In [91]:
# print unique values per column
remissions.nunique()

tkt_code               333804
order_date               1755
order_code                794
start_time              70338
truck_code                150
ship_plant_code             6
u_Volumen                  34
typed_time             352819
at_plant_time          352759
u_Cicle                   209
name                     2060
Nombre del proyecto     23561
ship_addr_line          73353
map_page                  953
dtype: int64

In [92]:
# Print order_code values that are duplicated to check for potential data entry errors
order_code_counts = remissions['order_code'].value_counts()
duplicate_order_codes = order_code_counts[order_code_counts > 1]
print("Duplicate order_code values and their counts:", duplicate_order_codes)

# print sum of duplicated order_code values to see how many rows are affected by potential data entry errors
total_duplicates = duplicate_order_codes.sum()
print(f"Total rows affected by duplicate order_code values: {total_duplicates}")

Duplicate order_code values and their counts: order_code
1113    1295
1086    1218
1070    1195
1107    1193
1147    1184
        ... 
1509       2
1515       2
1490       2
1492       2
1498       2
Name: count, Length: 754, dtype: int64
Total rows affected by duplicate order_code values: 353674


In [93]:
# print oldest typed_time row with order code 1113 and newest
order_1113 = remissions[remissions['order_code'] == '1113']
print("Oldest typed_time for order_code 1113:", order_1113['typed_time'].min())
print("Newest typed_time for order_code 1113:", order_1113['typed_time'].max())

Oldest typed_time for order_code 1113: 2020-02-05 08:25:55
Newest typed_time for order_code 1113: 2026-04-23 15:04:28


Order_code parece carecer de significado, se puede eliminar la columna sin problemas

In [94]:
# Drop order_code column since it means nothing relevant
remissions = remissions.drop(columns=['order_code'])
remissions.head(1)

,tkt_code,order_date,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
0,51013232,2020-02-04,2020-02-04 07:00:00,7044,510,6.0,2020-02-04 06:45:13,2020-02-04 08:17:24,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1


In [95]:
#Looks like the tkt_codes can be repeated after years of orders.
remissions[remissions['tkt_code'].duplicated(keep=False)].sort_values('tkt_code', ascending=True).head(6)


,tkt_code,order_date,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
1,51013233,2020-02-04,2020-02-04 07:00:00,6603,510,5.5,2020-02-04 06:45:22,2020-02-04 08:26:06,101,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
310446,51013233,2025-08-02,2025-08-02 07:00:00,9430,510,2.0,2025-08-02 06:46:30,2025-08-02 07:46:17,60,ONE TIME PROMOCIONES OCTAVIO RIOS,MARGARITO ROMERO,ING. CARRILLO 16344 COL TRAHUMARA,CHJ-3
3,51013236,2020-02-04,2020-02-04 08:15:00,6598,510,3.0,2020-02-04 08:02:36,2020-02-04 09:47:43,105,IVAN NOE SIMENTAL ORTEGA,COLECTOR SACRAMENTO,ARROLLO MIMBRE Y VIALIDAD SACRAMENTO,CH-J9
310448,51013236,2025-08-02,2025-08-02 07:00:00,7043,510,4.0,2025-08-02 06:54:06,2025-08-02 08:14:23,80,JONATHAN MARQUEZ BACA,OBRAS VARIAS,ARROYO NARAGUA #2225 LOS ARROYOS,CHD-5
5,51013239,2020-02-04,2020-02-04 09:00:00,6611,510,6.0,2020-02-04 08:36:24,2020-02-04 09:30:24,54,JULIO ARMANDO HINOJOS ENRIQUEZ,FRAC CALZADA DEL BOSQUE AGH,FRAC CALZADA DEL BOSQUE AGH FRAC CAL,CH-H3
310450,51013239,2025-08-02,2025-08-02 08:00:00,9430,510,6.5,2025-08-02 07:46:56,2025-08-02 09:03:00,77,FERRETERIA MOLINA DE CHIHUAHUA,FERRETERIA MOLINA DE CHIHUAHUA,MONTE HIMALAYA 4341 QUINTAS CAROLINA,CHH-8


We will now check the consistency of remissions per plant

In [96]:
# Lets see order ratios between plants
counts = remissions['ship_plant_code'].value_counts()
percentages = (counts / counts.sum()) * 100
print(counts, '\t', percentages.round(2)) # percentages
print("total:", counts.sum())

ship_plant_code
512    82647
510    71417
511    61024
515    52442
710    45897
514    40287
Name: count, dtype: int64 	 ship_plant_code
512    23.37
510    20.19
511    17.25
515    14.83
710    12.98
514    11.39
Name: count, dtype: float64
total: 353714


In [97]:
# See lowest and highest datetime for each plant
for plant_code in remissions['ship_plant_code'].unique():
    plant = remissions[remissions['ship_plant_code'] == plant_code]
    print(f"Plant {plant_code}:")
    print(f"  Lowest datetime: {plant['start_time'].min()}")
    print(f"  Highest datetime: {plant['start_time'].max()}")

Plant 510:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-04-24 09:30:00
Plant 511:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-04-24 10:00:00
Plant 512:
  Lowest datetime: 2020-02-04 05:00:00
  Highest datetime: 2026-04-24 11:30:00
Plant 515:
  Lowest datetime: 2020-02-04 08:30:00
  Highest datetime: 2026-04-24 10:00:00
Plant 710:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-04-24 10:15:00
Plant 514:
  Lowest datetime: 2022-07-20 07:00:00
  Highest datetime: 2026-04-24 10:45:00


In [98]:
# percentage of remissions that were not ordered between 8am and 6pm
remissions['hour'] = remissions['start_time'].dt.hour
outside_business_hours = remissions[(remissions['hour'] < 7) | (remissions['hour'] >= 18)]
percentage_outside = (outside_business_hours.shape[0] / remissions.shape[0])
print(f"Percentage of remissions ordered outside of business hours (7am-6pm): {percentage_outside * 100:.2f}%")
 

Percentage of remissions ordered outside of business hours (7am-6pm): 1.84%


Solo 1.84% son remisiones fuera del horario 7am - 6pm

In [99]:
# Visualize most common hours for remissions outside of business hours
outside_hours_dist = outside_business_hours['hour'].value_counts().sort_index()

fig = px.bar(
    x=outside_hours_dist.index,
    y=outside_hours_dist.values,
    title='Distribution of Remissions Outside Business Hours (Before 8am or After 6pm)',
    labels={'x': 'Hour of Day', 'y': 'Remission Count'},
    text=outside_hours_dist.values
)

fig.update_traces(textposition='outside', texttemplate='%{y}')
fig.update_layout(xaxis_tickmode='linear', xaxis_tick0=0, xaxis_dtick=1)
fig.show()

In [100]:
# Visualize most common hours for remissions during business hours
business_hours = remissions[(remissions['hour'] >= 7) & (remissions['hour'] < 18)]
business_hours_dist = business_hours['hour'].value_counts().sort_index()    
fig = px.bar(
    x=business_hours_dist.index,
    y=business_hours_dist.values,
    title='Distribution of Remissions During Business Hours (7am-6pm)',
    labels={'x': 'Hour of Day', 'y': 'Remission Count'},
    text=business_hours_dist.values
)
fig.update_traces(textposition='outside', texttemplate='%{y}')
fig.update_layout(xaxis_tickmode='linear', xaxis_tick0=0, xaxis_dtick=1)
fig.show()


In [101]:
# See hour range of remissions on saturdays
remissions['day_of_week'] = remissions['start_time'].dt.dayofweek
saturday_remissions = remissions[remissions['day_of_week'] == 5]
saturday_hours_dist = saturday_remissions['hour'].value_counts().sort_index()
fig = px.bar(
    x=saturday_hours_dist.index,
    y=saturday_hours_dist.values,
    title='Distribution of Remissions on Saturdays by Hour',
    labels={'x': 'Hour of Day', 'y': 'Remission Count'},
    text=saturday_hours_dist.values
)
fig.update_traces(textposition='outside', texttemplate='%{y}')
fig.update_layout(xaxis_tickmode='linear', xaxis_tick0=0, xaxis_dtick=1)
fig.show()

In [102]:
# Show table with percentages of remissions for each hour of the day, sorted by most common hours
hourly_dist = remissions['hour'].value_counts().sort_index().reset_index()
hourly_dist.columns = ['hour', 'count']
hourly_dist['percentage'] = (hourly_dist['count'] / hourly_dist['count'].sum() * 100).round(2)
hourly_dist = hourly_dist.sort_values('percentage', ascending=False).reset_index(drop=True)
print(hourly_dist)

    hour  count  percentage
0     10  44142       12.48
1     11  42880       12.12
2      9  41137       11.63
3     12  35055        9.91
4      8  32239        9.11
5     14  32133        9.08
6      7  31303        8.85
7     13  30859        8.72
8     15  30813        8.71
9     16  20292        5.74
10    17   6335        1.79
11     6   2847        0.80
12     5   1065        0.30
13    18    778        0.22
14    22    569        0.16
15    19    526        0.15
16     4    278        0.08
17    20    274        0.08
18     1    136        0.04
19     3     32        0.01
20     0      4        0.00
21     2      5        0.00
22    21      6        0.00
23    23      6        0.00


Hay más ordenes a las 5 de la mañana que a las 6 de la tarde, por lo que quitaré las 6pm de los horarios abiertos y dejare hasta las 5pm

#### Exporting Cleaned Dataset

In [103]:
# Has to be xlsx because of datetime format
remissions.to_excel("../data/processed/remissions_db_cleaned.xlsx", index=False)

### FOR THE NEXT PART

- We will add data_imputation using regressor models for the 2026 1 month gap in all plants and the 2024-2025 7 month gap for plant 710
 